In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain.vectorstores import FAISS
import json
import pandas as pd 

In [2]:
# ----------------------------
# 1. Load cached articles
# ----------------------------
CACHE_FILE = "news_cache.json" 

with open(CACHE_FILE, "r", encoding="utf-8") as f:
    articles_json = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(articles_json)
print(f"Loaded {len(df)} articles from cache.")

Loaded 99 articles from cache.


In [11]:
# ----------------------------
# 2. Combine text fields
# ----------------------------
def combine_article_text(row):
    title = row.get("title") or ""
    description = row.get("description") or ""
    content = row.get("content") or ""
    return f"{title} {description} {content}"

df["combined_text"] = df.apply(combine_article_text, axis=1)

In [12]:
# ----------------------------
# 3. Initialize OllamaEmbeddings
# ----------------------------
# If your TinyLlama model is running locally on a port (e.g., 11434)
# OllamaEmbeddings allows you to set base_url
embeddings = OllamaEmbeddings(
    model="tinyllama",
    base_url="http://host.docker.internal:11434"  # points to local machine from Docker
)


In [13]:
# ----------------------------
# 4. Create FAISS vector store
# ----------------------------
vector_db = FAISS.from_texts(df["combined_text"].tolist(), embeddings)
print("FAISS vector store created with", vector_db.index.ntotal, "vectors.")

FAISS vector store created with 99 vectors.


In [14]:
# ----------------------------
# 5. Example query
# ----------------------------
query = "Why does the term 'class action' appear frequently?"
results = vector_db.similarity_search(query, k=3)

for i, res in enumerate(results):
    print(f"{i+1}. {res.page_content}\n")

1. A US jury orders Google to pay $425M+ to plaintiffs in a 2020 class action lawsuit for collecting data even after they turned off Web & App Activity tracking This website is using a security service to protect itself from online attacks. The action you just performed triggered the security solution. There are several actions that could trigger this block including submitting a certain word or phrase, a SQL command… This website is using a security service to protect itself from online attacks. The action you just performed triggered the security solution. There are several actions that could trigger this block … [+157 chars]

2. It’s time for India to lure tempest tossed talent back home As Trump-rattled scientists and researchers in the US review their career options, we should act swiftly to attract them to India. Tamil Nadu has a plan for this that could be improved upon. What is common to Albert Einstein, James Franck, Erwin Schrödinger, Hans Bethe, Felix Bloch and Max Born, apa